# Project: Which study method works? Multi-armed bandits 🎰📚
Each evening you pick one study method (flashcards, re-reading, practice problems, explaining it to a rubber duck, a Manim video) and afterwards note whether the session "stuck" (1) or not (0). You don't know which is best, and every evening spent on a bad method is wasted: **explore vs. exploit**. This is RL with one state, the bandit problem.

Topic: [[RL Basics and MDPs]], [[Q-Learning and Policy Gradients]], [[Bayesian Inference]] · guide note: [[Project - Study Method Bandits]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

METHODS = ["re-reading", "flashcards", "practice problems", "rubber duck", "Manim video"]
TRUE_P = np.array([0.30, 0.50, 0.62, 0.55, 0.45])   # secret success rates (made up – in real life you never see these)

def run(policy, steps=1000, seed=0):
    rng = np.random.default_rng(seed); counts, rewards = np.zeros(5), np.zeros(5); regret = []
    for t in range(1, steps + 1):
        a = policy(counts, rewards, t, rng)
        r = rng.random() < TRUE_P[a]; counts[a] += 1; rewards[a] += r
        regret.append(TRUE_P.max() - TRUE_P[a])
    return np.cumsum(regret), counts

## 1. ε-greedy
With probability `eps` pick a random method, otherwise the best average so far (untried methods count as +∞ so each gets tried).

In [ ]:
def eps_greedy(counts, rewards, t, rng, eps=0.1):
    # TODO 1: explore with prob eps, else argmax of empirical means (untried = inf)
    raise NotImplementedError  # TODO 1

## 2. UCB1\nOptimism under uncertainty: try each arm once, then pick $\arg\max_a\ \hat\mu_a + \sqrt{2\ln t / n_a}$. Rarely-tried arms get a bonus.

In [ ]:
def ucb1(counts, rewards, t, rng):
    # TODO 2: untried arm first; else mean + sqrt(2 ln t / n)
    raise NotImplementedError  # TODO 2

## 3. Thompson sampling\nKeep a Beta posterior per arm: $\text{Beta}(1+\text{successes}, 1+\text{failures})$. Each step **sample** one plausible success rate per arm and pick the largest. Probability matching: an arm is chosen as often as it is likely to be the best.

In [ ]:
def thompson(counts, rewards, t, rng):
    # TODO 3: one Beta sample per arm, argmax
    raise NotImplementedError  # TODO 3

In [ ]:
def _t_pol():
    rng = np.random.default_rng(0); c = np.array([3, 3, 0, 3, 3.0]); r = np.array([1, 2, 0, 1, 1.0])
    ok1 = eps_greedy(c, r, 13, rng, eps=0.0) == 2 and eps_greedy(np.array([4, 4, 4, 4, 4.0]), np.array([1, 3, 2, 0, 0.0]), 21, rng, eps=0) == 1
    ok2 = ucb1(c, r, 13, rng) == 2 and ucb1(np.array([100, 1, 100, 100, 100.0]), np.array([60, 0, 50, 50, 50.0]), 401, rng) == 1
    picks = [thompson(np.array([50, 50, 0, 0, 0.0]), np.array([45, 5, 0, 0, 0.0]), 100, np.random.default_rng(s)) for s in range(200)]
    return ok1 and ok2 and picks.count(1) < 5 and picks.count(0) > 20
check("policies", _t_pol, "eps=0 must exploit (untried first); UCB must try unexplored/rarely-tried arms; Thompson must almost never pick an arm with 5/50 successes.")
if ready("policies"):
    pols = {"random": lambda c, r, t, g: int(g.integers(5)), "ε-greedy 0.1": eps_greedy, "UCB1": ucb1, "Thompson": thompson}
    final = {}
    for name, pol in pols.items():
        runs = [run(pol, seed=s) for s in range(30)]
        reg = np.mean([r[0] for r in runs], 0); final[name] = reg[-1]
        plt.plot(reg, label=f"{name} ({reg[-1]:.0f})")
    print({k: round(v, 1) for k, v in final.items()})
    plt.xlabel("evening"); plt.ylabel("cumulative regret"); plt.legend(); plt.show()
    print("evenings spent per method by Thompson (seed 0):", dict(zip(METHODS, run(thompson)[1].astype(int))))
    check("smart beats naive", lambda: final["Thompson"] < final["ε-greedy 0.1"] < final["random"] / 2 and final["UCB1"] < final["random"],
          "Expected Thompson < ε-greedy < half of random, and UCB1 below random.")
    print("→ UCB1's bonus √(2 ln t / n) is a worst-case bound: it keeps re-checking close arms (0.55 vs 0.62) for a long time. Its regret grows like log t, so it shines over long horizons.")

<details><summary>▶ Solution</summary>

```python
def eps_greedy(counts, rewards, t, rng, eps=0.1):
    if rng.random() < eps:
        return int(rng.integers(len(counts)))
    means = np.where(counts > 0, rewards / np.maximum(counts, 1), np.inf)
    return int(np.argmax(means))
```

```python
def ucb1(counts, rewards, t, rng):
    if np.any(counts == 0):
        return int(np.argmin(counts))
    return int(np.argmax(rewards / counts + np.sqrt(2 * np.log(t) / counts)))
```

```python
def thompson(counts, rewards, t, rng):
    return int(np.argmax(rng.beta(1 + rewards, 1 + counts - rewards)))
```
</details>

## Stretch goals
- **Use it for real:** log your own evenings in a TIL note (`method::`, `stuck::`) and run Thompson sampling on your data.
- Non-stationary arms (you get bored of flashcards): use a sliding window or discounting.
- Contextual bandits: the best method depends on the topic (maths vs. law), i.e. LinUCB.